In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from collections import Counter
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler 
from sklearn.utils import class_weight
from sklearn.preprocessing import LabelEncoder

In [2]:
data=pd.read_csv(f'/kaggle/input/datasets/jessemostipak/hotel-booking-demand/hotel_bookings.csv')
data.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
data=data.drop_duplicates()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87396 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87396 non-null  object 
 1   is_canceled                     87396 non-null  int64  
 2   lead_time                       87396 non-null  int64  
 3   arrival_date_year               87396 non-null  int64  
 4   arrival_date_month              87396 non-null  object 
 5   arrival_date_week_number        87396 non-null  int64  
 6   arrival_date_day_of_month       87396 non-null  int64  
 7   stays_in_weekend_nights         87396 non-null  int64  
 8   stays_in_week_nights            87396 non-null  int64  
 9   adults                          87396 non-null  int64  
 10  children                        87392 non-null  float64
 11  babies                          87396 non-null  int64  
 12  meal                            8739

In [4]:
data.drop(columns=['reservation_status_date','reservation_status'],inplace=True)

In [5]:
data.isna().sum()/len(data)
# country, agent, company has nan data
#company has 93% of nan data, which seems useless for model training
#next we should do eda to pick the features we need for model training

hotel                             0.000000
is_canceled                       0.000000
lead_time                         0.000000
arrival_date_year                 0.000000
arrival_date_month                0.000000
arrival_date_week_number          0.000000
arrival_date_day_of_month         0.000000
stays_in_weekend_nights           0.000000
stays_in_week_nights              0.000000
adults                            0.000000
children                          0.000046
babies                            0.000000
meal                              0.000000
country                           0.005172
market_segment                    0.000000
distribution_channel              0.000000
is_repeated_guest                 0.000000
previous_cancellations            0.000000
previous_bookings_not_canceled    0.000000
reserved_room_type                0.000000
assigned_room_type                0.000000
booking_changes                   0.000000
deposit_type                      0.000000
agent      

In [6]:
data['is_canceled'].astype('int')
country_df=data.groupby(['country'])['is_canceled'].sum()/data.groupby(['country'])['is_canceled'].count()
country_df.sort_values(ascending=False,inplace=True)
country_df.head(30)
#the country col is important
#since this col has few data missing, we will drop them directly

country
BEN    1.000000
MYT    1.000000
KHM    1.000000
JEY    1.000000
IMN    1.000000
HND    1.000000
NIC    1.000000
UMI    1.000000
VGB    1.000000
GLP    1.000000
GGY    1.000000
FJI    1.000000
HKG    0.913043
MAC    0.900000
ARE    0.854167
BHR    0.800000
MDV    0.750000
GEO    0.736842
GIB    0.733333
SEN    0.727273
AND    0.714286
QAT    0.714286
BGD    0.700000
SAU    0.666667
FRO    0.666667
PAK    0.615385
TZA    0.600000
PHL    0.593750
NGA    0.566667
AGO    0.564327
Name: is_canceled, dtype: float64

In [7]:
data = data.dropna(subset=['country'])
data.isna().sum()

hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_year                     0
arrival_date_month                    0
arrival_date_week_number              0
arrival_date_day_of_month             0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              4
babies                                0
meal                                  0
country                               0
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
deposit_type                          0
agent                             11868
company                           81852


In [8]:
country_counts = data['country'].value_counts()
valid_countries = country_counts[country_counts >= 100].index

data['country_group'] = data['country'].apply(
    lambda x: x if x in valid_countries else 'Other'
)

country_cancel_rate = data.groupby('country_group')['is_canceled'].mean()

def risk_bucket(rate):
    if rate > 0.5: return 'High'
    elif rate > 0.3: return 'Medium'
    else: return 'Low'

data['country_risk'] = data['country_group'].map(country_cancel_rate).apply(risk_bucket)

In [9]:
agent_df=data.groupby(['agent'])['is_canceled'].sum()/data.groupby(['agent'])['is_canceled'].count()
agent_df.sort_values(ascending=False,inplace=True)
agent_df.head(30)
#agent is important, but nan data cannot be deleted because there are too much of them
# we will fill -1 here to represent non-agent status

agent
55.0     1.000000
60.0     1.000000
403.0    1.000000
276.0    1.000000
93.0     1.000000
41.0     1.000000
257.0    1.000000
162.0    1.000000
236.0    1.000000
235.0    1.000000
483.0    1.000000
472.0    1.000000
492.0    1.000000
170.0    1.000000
323.0    1.000000
107.0    1.000000
291.0    1.000000
287.0    1.000000
286.0    1.000000
192.0    0.777778
58.0     0.729167
31.0     0.692308
531.0    0.650000
326.0    0.625000
4.0      0.611111
252.0    0.600000
214.0    0.600000
296.0    0.538462
440.0    0.521739
35.0     0.500000
Name: is_canceled, dtype: float64

In [10]:
data['agent'].fillna(-1)
data.isna().sum()

hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_year                     0
arrival_date_month                    0
arrival_date_week_number              0
arrival_date_day_of_month             0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              4
babies                                0
meal                                  0
country                               0
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
deposit_type                          0
agent                             11868
company                           81852



**Feature Engineering**

Feature choose

In [11]:
class_list=[]
for i in data.columns:
    if data[i].dtype=="object":
        class_list.append(i)
class_list        

['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type',
 'country_group',
 'country_risk']

In [12]:
to_remove=['arrival_date_month']
class_list=[col for col in class_list if col not in to_remove]
class_list

['hotel',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type',
 'country_group',
 'country_risk']

In [13]:
month_map = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12
}
data['arrival_date_month'] = data['arrival_date_month'].map(month_map)
data.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,country_group,country_risk
0,Resort Hotel,0,342,2015,7,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,PRT,Medium
1,Resort Hotel,0,737,2015,7,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,PRT,Medium
2,Resort Hotel,0,7,2015,7,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,GBR,Low
3,Resort Hotel,0,13,2015,7,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,GBR,Low
4,Resort Hotel,0,14,2015,7,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,GBR,Low


In [14]:
#next we will caculate the vars of cancellation rates of class cols to decide whici to drop
def get_cancel_variance(col):
    cancel_rate=data.groupby(col)['is_canceled'].mean()
    return cancel_rate.var()
    

In [15]:
cal_var={}
for col in class_list:
    if data[col].nunique()>100:
        cal_var[col] = 0
        continue
    cal_var[col]=get_cancel_variance(col)
cal_var

{'hotel': 0.0020038289027133264,
 'meal': 0.004305433897477014,
 'country': 0,
 'market_segment': 0.087506586250242,
 'distribution_channel': 0.07768554215735457,
 'reserved_room_type': 0.047970570574786105,
 'assigned_room_type': 0.10317973591089423,
 'deposit_type': 0.15956090273936685,
 'customer_type': 0.007538254484307856,
 'country_group': 0.009992824053128692,
 'country_risk': 0.03150906206976367}

In [16]:
var_thres=0.009
drop_class_list=[k for k, v in cal_var.items() if v<var_thres]
keep_class_list=[col for col in class_list if col not in drop_class_list]
keep_class_list.remove('country_group')
keep_class_list

['market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'country_risk']

In [17]:
drop_class_list
#to be accurate for dropping categorial cols, we will see the
#max differences of features in drop list

['hotel', 'meal', 'country', 'customer_type']

In [18]:
for col in drop_class_list :
    if col !='country':
        ct = pd.crosstab(data[col], data['is_canceled'], normalize='index')
        print(ct)
        cancel_rates = ct[1].values
        max_diff = cancel_rates.max() - cancel_rates.min()
        print(f'max difference of cancel rates:{max_diff}')
#turns out all should be kept

is_canceled          0         1
hotel                           
City Hotel    0.699656  0.300344
Resort Hotel  0.762962  0.237038
max difference of cancel rates:0.0633060645232876
is_canceled         0         1
meal                           
BB           0.733283  0.266717
FB           0.725000  0.275000
HB           0.729792  0.270208
SC           0.647245  0.352755
Undefined    0.831967  0.168033
max difference of cancel rates:0.18472212128448176
is_canceled             0         1
customer_type                      
Contract         0.836891  0.163109
Group            0.901304  0.098696
Transient        0.697637  0.302363
Transient-Party  0.847371  0.152629
max difference of cancel rates:0.20366616095179196


In [19]:
keep_class_list=keep_class_list+drop_class_list
keep_class_list

['market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'country_risk',
 'hotel',
 'meal',
 'country',
 'customer_type']

In [20]:
num_df=data.drop(columns=class_list+['country_group','country'])
num_df.head()

,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
0,0,342,2015,7,27,1,0,0,2,0.0,...,0,0,0,3,NaN,NaN,0,0.0,0,0
1,0,737,2015,7,27,1,0,0,2,0.0,...,0,0,0,4,NaN,NaN,0,0.0,0,0
2,0,7,2015,7,27,1,0,1,1,0.0,...,0,0,0,0,NaN,NaN,0,75.0,0,0
3,0,13,2015,7,27,1,0,1,1,0.0,...,0,0,0,0,304.0,NaN,0,75.0,0,0
4,0,14,2015,7,27,1,0,2,2,0.0,...,0,0,0,0,240.0,NaN,0,98.0,0,1


In [21]:
num_corr=num_df.corr()['is_canceled'].drop('is_canceled')
num_corr

lead_time                         0.183649
arrival_date_year                 0.087522
arrival_date_month                0.003143
arrival_date_week_number          0.000894
arrival_date_day_of_month         0.005202
stays_in_weekend_nights           0.059576
stays_in_week_nights              0.082798
adults                            0.080333
children                          0.067027
babies                           -0.020599
is_repeated_guest                -0.090154
previous_cancellations            0.050956
previous_bookings_not_canceled   -0.050558
booking_changes                  -0.094072
agent                            -0.043200
company                           0.006824
days_in_waiting_list              0.004297
adr                               0.126556
required_car_parking_spaces      -0.183715
total_of_special_requests        -0.121735
Name: is_canceled, dtype: float64

In [22]:
corr_threshold = 0.05
keep_num_list=num_corr[abs(num_corr)>=corr_threshold].index.tolist()
keep_num_list

['lead_time',
 'arrival_date_year',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests']

In [23]:
drop_num_list=num_corr[abs(num_corr)<corr_threshold].index.tolist()
drop_num_list

['arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'babies',
 'agent',
 'company',
 'days_in_waiting_list']

In [24]:
for col in drop_num_list:
    data[f"{col}_bin"] = pd.qcut(data[col], q=5, duplicates='drop')
    ct = pd.crosstab(data[f"{col}_bin"], data['is_canceled'], normalize='index')
    print(ct)
    cancel_rates = ct[1].values
    max_diff = cancel_rates.max() - cancel_rates.min()
    print(f"max differences of cancel rates of nums ：{max_diff:.2f}\n")
    data.drop(f"{col}_bin", axis=1, inplace=True)
    #we see the following cols should be kept: arrival_date_month、arrival_date_week_number、agent、company

is_canceled                    0         1
arrival_date_month_bin                    
(0.999, 3.0]            0.764057  0.235943
(3.0, 6.0]              0.699629  0.300371
(6.0, 8.0]              0.679492  0.320508
(8.0, 9.0]              0.754317  0.245683
(9.0, 12.0]             0.759901  0.240099
max differences of cancel rates of nums ：0.08

is_canceled                          0         1
arrival_date_week_number_bin                    
(0.999, 13.0]                 0.763579  0.236421
(13.0, 23.0]                  0.702246  0.297754
(23.0, 31.0]                  0.683721  0.316279
(31.0, 40.0]                  0.711769  0.288231
(40.0, 53.0]                  0.759903  0.240097
max differences of cancel rates of nums ：0.08

is_canceled                           0         1
arrival_date_day_of_month_bin                    
(0.999, 7.0]                   0.724408  0.275592
(7.0, 13.0]                    0.729246  0.270754
(13.0, 19.0]                   0.725303  0.274697
(19.0, 25.0]

In [25]:
keep_num_list=keep_num_list+['arrival_date_month','arrival_date_week_number','agent','company']
keep_num_list

['lead_time',
 'arrival_date_year',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'arrival_date_month',
 'arrival_date_week_number',
 'agent',
 'company']

In [26]:
df=data.copy()
keep_cols=keep_class_list+keep_num_list+ ['is_canceled']
df=df[keep_cols]
df.head()

,market_segment,distribution_channel,reserved_room_type,assigned_room_type,deposit_type,country_risk,hotel,meal,country,customer_type,...,previous_bookings_not_canceled,booking_changes,adr,required_car_parking_spaces,total_of_special_requests,arrival_date_month,arrival_date_week_number,agent,company,is_canceled
0,Direct,Direct,C,C,No Deposit,Medium,Resort Hotel,BB,PRT,Transient,...,0,3,0.0,0,0,7,27,NaN,NaN,0
1,Direct,Direct,C,C,No Deposit,Medium,Resort Hotel,BB,PRT,Transient,...,0,4,0.0,0,0,7,27,NaN,NaN,0
2,Direct,Direct,A,C,No Deposit,Low,Resort Hotel,BB,GBR,Transient,...,0,0,75.0,0,0,7,27,NaN,NaN,0
3,Corporate,Corporate,A,A,No Deposit,Low,Resort Hotel,BB,GBR,Transient,...,0,0,75.0,0,0,7,27,304.0,NaN,0
4,Online TA,TA/TO,A,A,No Deposit,Low,Resort Hotel,BB,GBR,Transient,...,0,0,98.0,0,1,7,27,240.0,NaN,0


In [27]:
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86944 entries, 0 to 119389
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   market_segment                  86944 non-null  object 
 1   distribution_channel            86944 non-null  object 
 2   reserved_room_type              86944 non-null  object 
 3   assigned_room_type              86944 non-null  object 
 4   deposit_type                    86944 non-null  object 
 5   country_risk                    86944 non-null  object 
 6   hotel                           86944 non-null  object 
 7   meal                            86944 non-null  object 
 8   country                         86944 non-null  object 
 9   customer_type                   86944 non-null  object 
 10  lead_time                       86944 non-null  int64  
 11  arrival_date_year               86944 non-null  int64  
 12  stays_in_weekend_nights         8694

**Model Build**


Data Split (train,test data)

In [28]:
Y = df['is_canceled']
Y.shape

(86944,)

In [29]:
X=df.drop(columns='is_canceled')
X.shape

(86944, 27)

In [30]:
X.columns

Index(['market_segment', 'distribution_channel', 'reserved_room_type',
       'assigned_room_type', 'deposit_type', 'country_risk', 'hotel', 'meal',
       'country', 'customer_type', 'lead_time', 'arrival_date_year',
       'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'booking_changes', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'arrival_date_month', 'arrival_date_week_number', 'agent', 'company'],
      dtype='object')

In [31]:
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=33)

In [32]:
from sklearn.preprocessing import OrdinalEncoder
encoders = {}
for col in keep_class_list:
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    x_train[col] = enc.fit_transform(x_train[[col]])
    x_test[col] = enc.transform(x_test[[col]])  
    encoders[col] = enc

In [33]:
estimator=xgb.XGBClassifier(objective='binary:logistic',random_state=23)

In [34]:
params_dict={'n_estimators': [100,200,300],
    'max_depth': [3, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
}
grid_model=GridSearchCV(estimator, param_grid=params_dict, cv=4)
grid_model.fit(x_train,y_train)
best_core_params=grid_model.best_params_
print(best_core_params)


{'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 300}


In [35]:
estimator= xgb.XGBClassifier(**best_core_params)
estimator.fit(x_train,y_train)
y_pre=estimator.predict(x_test)
print(f"prediction result：{y_pre}")
print(f"predict accuracy：{accuracy_score(y_test,y_pre)}")

prediction result：[0 0 0 ... 0 0 0]
predict accuracy：0.8567485191787911


In [36]:
# final_params = {
#     'n_estimators': best_core_params['n_estimators'],
#     'max_depth': best_core_params['max_depth'],
#     'learning_rate': best_core_params['learning_rate'],
#     'subsample': 0.8,          
#     'colsample_bytree': 0.8,   
#     'gamma': 1,               
#     'min_child_weight': 3,    
#     'random_state': 42,
#     'scale_pos_weight': 3,    
#     'eval_metric': 'logloss'
# }
# estimator2=xgb.XGBClassifier(**final_params)
# estimator2.fit(x_train,y_train)
# y_pre=estimator2.predict(x_test)
# print(f"prediction result：{y_pre}")
# print(f"predict accuracy：{accuracy_score(y_test,y_pre)}")

In [37]:
print(f"classification report：{classification_report(y_test,y_pre)}")

classification report：              precision    recall  f1-score   support

           0       0.89      0.91      0.90     12585
           1       0.76      0.71      0.73      4804

    accuracy                           0.86     17389
   macro avg       0.82      0.81      0.82     17389
weighted avg       0.85      0.86      0.86     17389

